In [34]:
# !pip install langchain langchain-core langchain-openai langchain_community pypdf langchain-text-splitters faiss-cpu

In [17]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [4]:
documents = []

for file in os.listdir('./Documents/'):
    if file.endswith(".pdf"):
        pdf_path = os.path.join('./Documents/', file)
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        documents.extend(docs)

In [5]:
print(f"Loaded {len(documents)} pages from PDFs")

Loaded 28 pages from PDFs


In [6]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [7]:
split_docs = text_splitter.split_documents(documents)

In [8]:
print(f"Split into {len(split_docs)} chunks")

Split into 28 chunks


### Embedding Generation and Storing in Vector Store

In [9]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(split_docs, embeddings)

In [10]:
vector_store.index_to_docstore_id

{0: '468a2433-24db-4f22-b298-cd4b3075cf3d',
 1: '5a565356-96f1-469b-b10f-89fff9437368',
 2: 'da46aa21-f2be-428a-bf3c-5f46de7468c9',
 3: '3a4f5baf-aaf4-4270-bef1-6f028cccba3c',
 4: '49771b7d-ed0c-4738-8c60-31347b70de62',
 5: '57aa0b9d-f668-4e1b-909d-1736950ae476',
 6: 'e323e072-f27f-4809-ae63-137cfc291cb2',
 7: 'ea4c1f45-6f3c-4245-84fd-d06ee9a61396',
 8: 'b5408014-1f77-4947-967d-a6f211a1093d',
 9: '3201ec94-d5a8-4dce-9323-2f0704c21e0f',
 10: 'c04e16f4-17cc-4acd-9bdb-d4460f748570',
 11: 'd91cd40e-e5ba-48b9-b1ef-e7815c750bf2',
 12: '4a5fa351-e2b3-4bb9-be84-aabd98b22711',
 13: 'db1688af-f995-4731-8585-a21459e3e429',
 14: 'bc20d26d-3580-446e-8e8e-c119342c089e',
 15: '6716e212-6a35-4f51-bf86-85fa84f7c09d',
 16: '7fec6d4c-3848-4f6f-8751-31c7573c52ec',
 17: '48b6c3d6-7fb3-4cf9-b1e1-17033a9a0616',
 18: '39333cff-6561-489e-b0ff-2411ccca42fb',
 19: '0c5500e1-d9a3-473f-9a96-1e5cd270578a',
 20: 'e35c7f0c-553e-4e26-aac5-5821037b1a06',
 21: 'c05d073d-ffa6-48d5-84c8-7f24c4bb144b',
 22: '981606b9-8dcd-

### Retrieval

In [11]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [12]:
retriever.invoke('What is motto of PaperNest?')

[Document(id='5a565356-96f1-469b-b10f-89fff9437368', metadata={'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX via pandoc', 'creationdate': '2026-02-24T11:57:16+00:00', 'source': './Documents/About Us.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Our Vision\nTo become a trusted neighborhood and online destination for premium sta-\ntionery, empowering ideas one page at a time.\nAt PaperNest Stationery Co. , we don’t just sell paper — we support ideas,\ndreams, and ambitions.\nLet’s write something meaningful together.\n2'),
 Document(id='468a2433-24db-4f22-b298-cd4b3075cf3d', metadata={'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX via pandoc', 'creationdate': '2026-02-24T11:57:16+00:00', 'source': './Documents/About Us.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='About Us – PaperNest Stationery Co.\nAt PaperNest Stationery Co. , we believe every great idea begins on paper.\nFounded with a passion for creativity and organizatio

### Augmentation

In [15]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [18]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided documents context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

### Generation

In [25]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [26]:
parser = StrOutputParser()

In [29]:
def get_context(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

In [32]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(get_context),
    'question': RunnablePassthrough()
})

In [34]:
main_chain = parallel_chain | prompt | llm | parser

In [37]:
main_chain.invoke('what is the mission and vision of this company?')

'The vision of PaperNest Stationery Co. is to become a trusted neighborhood and online destination for premium stationery, empowering ideas one page at a time. The mission is to support ideas, dreams, and ambitions through the sale of paper, encouraging meaningful writing.'

In [38]:
main_chain.invoke('what is the leave policy?')

'The leave policy includes the following types of leave:\n\n1. **Casual Leave**: For personal reasons.\n2. **Sick Leave**: For health-related matters, with medical proof required for extended leave.\n3. **Earned/Annual Leave**: Granted after completion of probation, as per company policy.\n4. **Public Holidays**: As declared by the company in compliance with government regulations.\n\nLeave must be approved in advance, except in emergencies.'